In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sqlite3
from scipy.stats import ttest_ind
import scipy.stats as stats
warnings.filterwarnings('ignore' )

In [2]:
# creating database connection
conn = sqlite3.connect('inventory.db')

# fetching vendor summary data
df = pd.read_sql_query("select * from vendor_sales_summary", conn)
df.head()

,VendorNumber,VendorName,Brand,Description,PurchasePrice,Volume,ActualPrice,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesPrice,TotalSalesDollars,TotalExciseTax,TotalSalesQuantity,FreightCost
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,1750.0,36.99,145080,3811251.60,672819.31,5101919.51,260999.20,142049.0,68601.68
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,1750.0,28.99,164038,3804041.22,561512.37,4819073.49,294438.66,160247.0,144929.24
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,1750.0,24.99,187407,3418303.68,461140.15,4538120.60,343854.07,187140.0,123780.22
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,1750.0,22.99,201682,3261197.94,420050.01,4475972.88,368242.80,200412.0,257032.07
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,1750.0,29.99,138109,3023206.01,545778.28,4223107.62,249587.83,135838.0,257032.07


In [ ]:
# creating database connection
conn = sqlite3.connect('inventory.db')

# fetching vendor summary data
df = pd.read_sql_query("select * from vendor_sales_summary", conn)

# Data quality check
print(f"Dataset Shape: {df.shape}")
print(f"Date Range: {df.describe().T[['min', 'max']].head()}")

# Basic statistics
print("\nKey Business Metrics:")
print(f"Total Vendors: {df['VendorNumber'].nunique()}")
print(f"Total Brands: {df['Brand'].nunique()}")
print(f"Total Sales: ${df['TotalSalesDollars'].sum():,.2f}")
print(f"Total Purchases: ${df['TotalPurchaseDollars'].sum():,.2f}")
print(f"Total Gross Profit: ${(df['TotalSalesDollars'] - df['TotalPurchaseDollars']).sum():,.2f}")

df.head()

In [ ]:
# Data Cleaning and Feature Engineering
def clean_and_enhance_data(df):
    """Clean data and create business-critical features"""
    
    # Handle missing values
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        df[col] = df[col].fillna(0)
    
    # Create derived business metrics
    df['GrossProfit'] = df['TotalSalesDollars'] - df['TotalPurchaseDollars']
    df['ProfitMargin'] = (df['GrossProfit'] / df['TotalSalesDollars']) * 100
    df['StockTurnover'] = df['TotalSalesQuantity'] / df['TotalPurchaseQuantity']
    df['SalesToPurchaseRatio'] = df['TotalSalesDollars'] / df['TotalPurchaseDollars']
    df['UnitCost'] = df['TotalPurchaseDollars'] / df['TotalPurchaseQuantity']
    df['UnitRevenue'] = df['TotalSalesDollars'] / df['TotalSalesQuantity']
    df['FreightRatio'] = (df['FreightCost'] / df['TotalPurchaseDollars']) * 100
    
    # Handle infinite values and division by zero
    df.replace([np.inf, -np.inf], 0, inplace=True)
    df.fillna(0, inplace=True)
    
    # Clean categorical columns
    df['VendorName'] = df['VendorName'].str.strip()
    df['Description'] = df['Description'].str.strip()
    
    return df

# Apply cleaning
df_clean = clean_and_enhance_data(df)

# Filter for meaningful analysis (exclude records with no sales activity)
df_analysis = df_clean[
    (df_clean['TotalSalesDollars'] > 0) & 
    (df_clean['TotalPurchaseDollars'] > 0) & 
    (df_clean['ProfitMargin'] > -100) &
    (df_clean['ProfitMargin'] < 100)
].copy()

print(f"Original records: {len(df)}")
print(f"Analysis records: {len(df_analysis)}")
print(f"Data quality improvement: {((len(df) - len(df_analysis)) / len(df)) * 100:.1f}% removal rate")

df_analysis.head()

In [ ]:
# Exploratory Data Analysis - Summary Statistics
print("=== SUMMARY STATISTICS ===")
summary_stats = df_analysis.describe().T
print(summary_stats)

# Key Business Insights
print("\n=== KEY BUSINESS INSIGHTS ===")
print(f"Average Profit Margin: {df_analysis['ProfitMargin'].mean():.2f}%")
print(f"Median Stock Turnover: {df_analysis['StockTurnover'].median():.2f}")
print(f"Average Freight Cost Ratio: {df_analysis['FreightRatio'].mean():.2f}%")

# Identify top and bottom performers
top_vendors = df_analysis.groupby('VendorName')['TotalSalesDollars'].sum().sort_values(ascending=False).head(10)
bottom_vendors = df_analysis.groupby('VendorName')['TotalSalesDollars'].sum().sort_values().head(10)

print(f"\nTop 10 Vendors by Sales:")
for vendor, sales in top_vendors.items():
    print(f"{vendor}: ${sales:,.2f}")

print(f"\nBottom 10 Vendors by Sales:")
for vendor, sales in bottom_vendors.items():
    print(f"{vendor}: ${sales:,.2f}")

In [ ]:
# Visualization Setup
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Distribution Analysis
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Key Business Metrics Distribution', fontsize=16, fontweight='bold')

# Profit Margin Distribution
axes[0,0].hist(df_analysis['ProfitMargin'], bins=50, alpha=0.7, edgecolor='black')
axes[0,0].set_title('Profit Margin Distribution (%)')
axes[0,0].set_xlabel('Profit Margin %')
axes[0,0].set_ylabel('Frequency')
axes[0,0].axvline(df_analysis['ProfitMargin'].mean(), color='red', linestyle='--', label=f'Mean: {df_analysis["ProfitMargin"].mean():.1f}%')
axes[0,0].legend()

# Stock Turnover Distribution
axes[0,1].hist(df_analysis['StockTurnover'], bins=50, alpha=0.7, edgecolor='black')
axes[0,1].set_title('Stock Turnover Distribution')
axes[0,1].set_xlabel('Stock Turnover Ratio')
axes[0,1].set_ylabel('Frequency')
axes[0,1].axvline(df_analysis['StockTurnover'].mean(), color='red', linestyle='--', label=f'Mean: {df_analysis["StockTurnover"].mean():.2f}')
axes[0,1].legend()

# Sales Dollars Distribution
axes[0,2].hist(df_analysis['TotalSalesDollars'], bins=50, alpha=0.7, edgecolor='black')
axes[0,2].set_title('Sales Dollars Distribution')
axes[0,2].set_xlabel('Total Sales Dollars')
axes[0,2].set_ylabel('Frequency')
axes[0,2].axvline(df_analysis['TotalSalesDollars'].mean(), color='red', linestyle='--', label=f'Mean: ${df_analysis["TotalSalesDollars"].mean():,.0f}')
axes[0,2].legend()

# Freight Cost Distribution
axes[1,0].hist(df_analysis['FreightCost'], bins=50, alpha=0.7, edgecolor='black')
axes[1,0].set_title('Freight Cost Distribution')
axes[1,0].set_xlabel('Freight Cost ($)')
axes[1,0].set_ylabel('Frequency')
axes[1,0].axvline(df_analysis['FreightCost'].mean(), color='red', linestyle='--', label=f'Mean: ${df_analysis["FreightCost"].mean():,.0f}')
axes[1,0].legend()

# Gross Profit Distribution
axes[1,1].hist(df_analysis['GrossProfit'], bins=50, alpha=0.7, edgecolor='black')
axes[1,1].set_title('Gross Profit Distribution')
axes[1,1].set_xlabel('Gross Profit ($)')
axes[1,1].set_ylabel('Frequency')
axes[1,1].axvline(df_analysis['GrossProfit'].mean(), color='red', linestyle='--', label=f'Mean: ${df_analysis["GrossProfit"].mean():,.0f}')
axes[1,1].legend()

# Unit Cost Distribution
axes[1,2].hist(df_analysis['UnitCost'], bins=50, alpha=0.7, edgecolor='black')
axes[1,2].set_title('Unit Cost Distribution')
axes[1,2].set_xlabel('Unit Cost ($)')
axes[1,2].set_ylabel('Frequency')
axes[1,2].axvline(df_analysis['UnitCost'].mean(), color='red', linestyle='--', label=f'Mean: ${df_analysis["UnitCost"].mean():.2f}')
axes[1,2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Research Question 1: Identify brands needing promotional and pricing adjustments
def identify_promotional_targets(df):
    """Identify brands with low sales but high profit margins"""
    
    brand_performance = df.groupby('Description').agg({
        'TotalSalesDollars': 'sum',
        'ProfitMargin': 'mean',
        'TotalSalesQuantity': 'sum',
        'GrossProfit': 'sum'
    }).reset_index()
    
    # Set thresholds
    low_sales_threshold = brand_performance['TotalSalesDollars'].quantile(0.15)
    high_margin_threshold = brand_performance['ProfitMargin'].quantile(0.85)
    
    # Identify target brands
    target_brands = brand_performance[
        (brand_performance['TotalSalesDollars'] < low_sales_threshold) &
        (brand_performance['ProfitMargin'] > high_margin_threshold)
    ].sort_values('ProfitMargin', ascending=False)
    
    return brand_performance, target_brands

brand_perf, target_brands = identify_promotional_targets(df_analysis)

print(f"=== BRANDS NEEDING PROMOTIONAL ADJUSTMENTS ===")
print(f"Total brands analyzed: {len(brand_perf)}")
print(f"Target brands identified: {len(target_brands)}")
print(f"Low sales threshold: ${brand_perf['TotalSalesDollars'].quantile(0.15):,.2f}")
print(f"High margin threshold: {brand_perf['ProfitMargin'].quantile(0.85):.2f}%")

print("\nTop 10 Target Brands for Promotion:")
print(target_brands.head(10)[['Description', 'TotalSalesDollars', 'ProfitMargin', 'GrossProfit']].to_string(index=False))

# Visualization
plt.figure(figsize=(12, 8))
plt.scatter(brand_perf['TotalSalesDollars'], brand_perf['ProfitMargin'], alpha=0.6, s=50)
plt.scatter(target_brands['TotalSalesDollars'], target_brands['ProfitMargin'], color='red', s=100, alpha=0.8, label='Target Brands')
plt.axvline(low_sales_threshold, color='orange', linestyle='--', alpha=0.7, label='Low Sales Threshold')
plt.axhline(high_margin_threshold, color='green', linestyle='--', alpha=0.7, label='High Margin Threshold')
plt.xlabel('Total Sales Dollars ($)')
plt.ylabel('Profit Margin (%)')
plt.title('Brand Performance: Sales vs Profit Margin')
plt.legend()
plt.xscale('log')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Research Question 2: Top vendors and brands by sales performance
def analyze_top_performers(df):
    """Analyze top performing vendors and brands"""
    
    # Top vendors by sales
    top_vendors = df.groupby('VendorName').agg({
        'TotalSalesDollars': 'sum',
        'TotalPurchaseDollars': 'sum',
        'GrossProfit': 'sum',
        'ProfitMargin': 'mean',
        'StockTurnover': 'mean'
    }).sort_values('TotalSalesDollars', ascending=False).head(10)
    
    # Top brands by sales
    top_brands = df.groupby('Description').agg({
        'TotalSalesDollars': 'sum',
        'TotalPurchaseDollars': 'sum',
        'GrossProfit': 'sum',
        'ProfitMargin': 'mean',
        'TotalSalesQuantity': 'sum'
    }).sort_values('TotalSalesDollars', ascending=False).head(10)
    
    return top_vendors, top_brands

top_vendors, top_brands = analyze_top_performers(df_analysis)

print("=== TOP 10 VENDORS BY SALES ===")
print(top_vendors[['TotalSalesDollars', 'GrossProfit', 'ProfitMargin']].round(2))

print("\n=== TOP 10 BRANDS BY SALES ===")
print(top_brands[['TotalSalesDollars', 'GrossProfit', 'ProfitMargin']].round(2))

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Top Vendors
top_vendors['TotalSalesDollars'].plot(kind='bar', ax=ax1, color='skyblue')
ax1.set_title('Top 10 Vendors by Sales')
ax1.set_ylabel('Sales Dollars ($)')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3)

# Format y-axis to show values in millions
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

# Top Brands
top_brands['TotalSalesDollars'].plot(kind='bar', ax=ax2, color='lightcoral')
ax2.set_title('Top 10 Brands by Sales')
ax2.set_ylabel('Sales Dollars ($)')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True, alpha=0.3)

# Format y-axis to show values in millions
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

plt.tight_layout()
plt.show()

In [ ]:
# Research Question 3: Vendor purchase contribution analysis
def analyze_purchase_contribution(df):
    """Analyze vendor contribution to total purchases"""
    
    vendor_performance = df.groupby('VendorName').agg({
        'TotalPurchaseDollars': 'sum',
        'TotalSalesDollars': 'sum',
        'GrossProfit': 'sum'
    }).sort_values('TotalPurchaseDollars', ascending=False)
    
    # Calculate contribution percentage
    total_purchases = vendor_performance['TotalPurchaseDollars'].sum()
    vendor_performance['PurchaseContribution'] = (vendor_performance['TotalPurchaseDollars'] / total_purchases) * 100
    vendor_performance['SalesContribution'] = (vendor_performance['TotalSalesDollars'] / vendor_performance['TotalSalesDollars'].sum()) * 100
    
    # Calculate cumulative contribution
    vendor_performance['CumulativeContribution'] = vendor_performance['PurchaseContribution'].cumsum()
    
    return vendor_performance

vendor_contrib = analyze_purchase_contribution(df_analysis)

print("=== VENDOR PURCHASE CONTRIBUTION ANALYSIS ===")
top_10_contributors = vendor_contrib.head(10)
print(top_10_contributors[['TotalPurchaseDollars', 'PurchaseContribution', 'SalesContribution']].round(2))

print(f"\nTop 10 vendors contribute {vendor_contrib.head(10)['PurchaseContribution'].sum():.2f}% of total purchases")
print(f"Remaining {len(vendor_contrib) - 10} vendors contribute {100 - vendor_contrib.head(10)['PurchaseContribution'].sum():.2f}% of total purchases")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Purchase Contribution Bar Chart
top_10_contributors['PurchaseContribution'].plot(kind='bar', ax=ax1, color='gold')
ax1.set_title('Top 10 Vendors - Purchase Contribution (%)')
ax1.set_ylabel('Contribution Percentage')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3)

# Cumulative Contribution Line Chart
ax2.plot(range(len(vendor_contrib)), vendor_contrib['CumulativeContribution'], 'b-', linewidth=2)
ax2.axhline(y=80, color='red', linestyle='--', alpha=0.7, label='80% Threshold')
ax2.axvline(x=vendor_contrib[vendor_contrib['CumulativeContribution'] >= 80].index[0], 
            color='red', linestyle='--', alpha=0.7)
ax2.set_title('Cumulative Purchase Contribution')
ax2.set_xlabel('Vendor Rank')
ax2.set_ylabel('Cumulative Contribution (%)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Research Question 4: Bulk purchasing impact on unit cost
def analyze_bulk_purchasing(df):
    """Analyze the impact of bulk purchasing on unit costs"""
    
    # Create order size categories
    df_analysis['OrderSizeCategory'] = pd.qcut(df_analysis['TotalPurchaseQuantity'], 
                                             q=3, 
                                             labels=['Small', 'Medium', 'Large'])
    
    # Calculate unit purchase price
    df_analysis['UnitPurchasePrice'] = df_analysis['TotalPurchaseDollars'] / df_analysis['TotalPurchaseQuantity']
    
    # Analyze unit cost by order size
    bulk_analysis = df_analysis.groupby('OrderSizeCategory').agg({
        'UnitPurchasePrice': 'mean',
        'TotalPurchaseQuantity': 'mean',
        'TotalPurchaseDollars': 'sum',
        'ProfitMargin': 'mean'
    }).round(2)
    
    print("=== BULK PURCHASING ANALYSIS ===")
    print(bulk_analysis)
    
    # Calculate cost reduction percentage
    small_order_cost = bulk_analysis.loc['Small', 'UnitPurchasePrice']
    large_order_cost = bulk_analysis.loc['Large', 'UnitPurchasePrice']
    cost_reduction = ((small_order_cost - large_order_cost) / small_order_cost) * 100
    
    print(f"\nCost reduction from Small to Large orders: {cost_reduction:.2f}%")
    print(f"Price difference: ${small_order_cost - large_order_cost:.2f} per unit")
    
    return bulk_analysis, cost_reduction

bulk_analysis, cost_reduction = analyze_bulk_purchasing(df_analysis)

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Unit Cost by Order Size
bulk_analysis['UnitPurchasePrice'].plot(kind='bar', ax=ax1, color=['lightgreen', 'orange', 'red'])
ax1.set_title('Average Unit Cost by Order Size')
ax1.set_ylabel('Unit Cost ($)')
ax1.set_xlabel('Order Size Category')
ax1.tick_params(axis='x', rotation=0)
ax1.grid(True, alpha=0.3)

# Box plot for unit costs by order size
order_sizes = df_analysis['OrderSizeCategory'].unique()
unit_costs_by_size = [df_analysis[df_analysis['OrderSizeCategory'] == size]['UnitPurchasePrice'] 
                      for size in order_sizes]

ax2.boxplot(unit_costs_by_size, labels=order_sizes)
ax2.set_title('Unit Cost Distribution by Order Size')
ax2.set_ylabel('Unit Cost ($)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Research Question 5: Inventory turnover analysis
def analyze_inventory_turnover(df):
    """Analyze vendors with low inventory turnover"""
    
    # Identify vendors with low turnover
    low_turnover_vendors = df[df['StockTurnover'] < 1].copy()
    
    if len(low_turnover_vendors) > 0:
        # Aggregate by vendor
        vendor_turnover = low_turnover_vendors.groupby('VendorName').agg({
            'StockTurnover': 'mean',
            'TotalPurchaseQuantity': 'sum',
            'TotalSalesQuantity': 'sum',
            'TotalPurchaseDollars': 'sum',
            'GrossProfit': 'sum'
        }).sort_values('StockTurnover', ascending=True).head(10)
        
        print("=== VENDORS WITH LOW INVENTORY TURNOVER ===")
        print(vendor_turnover.round(3))
        
        # Calculate unsold inventory value
        low_turnover_vendors['UnsoldInventoryValue'] = (
            (low_turnover_vendors['TotalPurchaseQuantity'] - low_turnover_vendors['TotalSalesQuantity']) * 
            low_turnover_vendors['UnitCost']
        )
        
        total_unsold_value = low_turnover_vendors['UnsoldInventoryValue'].sum()
        print(f"\nTotal capital locked in unsold inventory: ${total_unsold_value:,.2f}")
        
        # Top contributors to unsold inventory
        unsold_by_vendor = low_turnover_vendors.groupby('VendorName')['UnsoldInventoryValue'].sum().sort_values(ascending=False).head(10)
        print("\nTop 10 vendors with highest unsold inventory value:")
        print(unsold_by_vendor.round(2))
        
        return vendor_turnover, unsold_by_vendor, total_unsold_value
    else:
        print("No vendors with stock turnover below 1 found.")
        return None, None, 0

vendor_turnover, unsold_by_vendor, total_unsold = analyze_inventory_turnover(df_analysis)

# Visualization if data exists
if vendor_turnover is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    
    # Low Turnover Vendors
    vendor_turnover['StockTurnover'].plot(kind='bar', ax=ax1, color='crimson')
    ax1.set_title('Vendors with Lowest Inventory Turnover')
    ax1.set_ylabel('Stock Turnover Ratio')
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(True, alpha=0.3)
    
    # Unsold Inventory Value
    unsold_by_vendor.plot(kind='bar', ax=ax2, color='darkred')
    ax2.set_title('Unsold Inventory Value by Vendor')
    ax2.set_ylabel('Unsold Inventory Value ($)')
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(True, alpha=0.3)
    
    # Format y-axis to show values in thousands
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e3:.0f}K'))
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Research Question 6: Statistical Analysis - Profit Margin Differences
def statistical_profit_analysis(df):
    """Perform statistical analysis on profit margins between top and low performing vendors"""
    
    # Define performance thresholds
    sales_75th = df['TotalSalesDollars'].quantile(0.75)
    sales_25th = df['TotalSalesDollars'].quantile(0.25)
    
    # Classify vendors
    top_performers = df[df['TotalSalesDollars'] >= sales_75th]['ProfitMargin']
    low_performers = df[df['TotalSalesDollars'] <= sales_25th]['ProfitMargin']
    
    print("=== STATISTICAL ANALYSIS OF PROFIT MARGINS ===")
    print(f"Top performers threshold: ${sales_75th:,.2f}")
    print(f"Low performers threshold: ${sales_25th:,.2f}")
    print(f"Top performers count: {len(top_performers)}")
    print(f"Low performers count: {len(low_performers)}")
    
    # Descriptive statistics
    print(f"\nTop Performers - Profit Margin:")
    print(f"Mean: {top_performers.mean():.2f}%")
    print(f"Std: {top_performers.std():.2f}%")
    print(f"Median: {top_performers.median():.2f}%")
    
    print(f"\nLow Performers - Profit Margin:")
    print(f"Mean: {low_performers.mean():.2f}%")
    print(f"Std: {low_performers.std():.2f}%")
    print(f"Median: {low_performers.median():.2f}%")
    
    # Confidence intervals
    def confidence_interval(data, confidence=0.95):
        n = len(data)
        mean = np.mean(data)
        std_err = stats.sem(data)
        h = std_err * stats.t.ppf((1 + confidence) / 2., n-1)
        return mean, mean - h, mean + h
    
    top_mean, top_lower, top_upper = confidence_interval(top_performers)
    low_mean, low_lower, low_upper = confidence_interval(low_performers)
    
    print(f"\n95% Confidence Intervals:")
    print(f"Top Performers: [{top_lower:.2f}%, {top_upper:.2f}%]")
    print(f"Low Performers: [{low_lower:.2f}%, {low_upper:.2f}%]")
    
    # T-test
    t_stat, p_value = ttest_ind(top_performers, low_performers)
    print(f"\nT-test Results:")
    print(f"T-statistic: {t_stat:.4f}")
    print(f"P-value: {p_value:.4f}")
    print(f"Significant difference: {'Yes' if p_value < 0.05 else 'No'}")
    
    return top_performers, low_performers, (top_mean, top_lower, top_upper), (low_mean, low_lower, low_upper)

top_perf, low_perf, top_ci, low_ci = statistical_profit_analysis(df_analysis)

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Histogram comparison
ax1.hist(top_perf, bins=30, alpha=0.7, label='Top Performers', color='blue', density=True)
ax1.hist(low_perf, bins=30, alpha=0.7, label='Low Performers', color='red', density=True)
ax1.axvline(top_ci[0], color='blue', linestyle='--', alpha=0.8)
ax1.axvline(low_ci[0], color='red', linestyle='--', alpha=0.8)
ax1.set_title('Profit Margin Distribution: Top vs Low Performers')
ax1.set_xlabel('Profit Margin (%)')
ax1.set_ylabel('Density')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Confidence intervals plot
categories = ['Top Performers', 'Low Performers']
means = [top_ci[0], low_ci[0]]
errors = [[top_ci[0]-top_ci[1], low_ci[0]-low_ci[1]], [top_ci[2]-top_ci[0], low_ci[2]-low_ci[0]]]

ax2.errorbar(categories, means, yerr=errors, fmt='o', capsize=10, capthick=2, markersize=8)
ax2.set_title('95% Confidence Intervals for Profit Margins')
ax2.set_ylabel('Profit Margin (%)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation Analysis and Heatmap
def correlation_analysis(df):
    """Perform correlation analysis on key business metrics"""
    
    # Select key numerical columns for correlation
    key_metrics = ['TotalPurchaseDollars', 'TotalSalesDollars', 'GrossProfit', 'ProfitMargin', 
                   'StockTurnover', 'FreightCost', 'UnitCost', 'UnitRevenue', 'TotalPurchaseQuantity', 
                   'TotalSalesQuantity', 'SalesToPurchaseRatio']
    
    correlation_matrix = df[key_metrics].corr()
    
    print("=== CORRELATION ANALYSIS ===")
    print("Key Insights:")
    
    # Find strongest correlations
    corr_pairs = []
    for i in range(len(key_metrics)):
        for j in range(i+1, len(key_metrics)):
            corr_val = correlation_matrix.iloc[i, j]
            if abs(corr_val) > 0.3:  # Only show meaningful correlations
                corr_pairs.append((key_metrics[i], key_metrics[j], corr_val))
    
    # Sort by absolute correlation value
    corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
    
    print("\nStrongest Correlations:")
    for metric1, metric2, corr in corr_pairs[:10]:
        print(f"{metric1} ↔ {metric2}: {corr:.3f}")
    
    return correlation_matrix

corr_matrix = correlation_analysis(df_analysis)

# Create heatmap
plt.figure(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": .8}, fmt='.2f')
plt.title('Correlation Matrix of Key Business Metrics', fontsize=16, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Executive Summary and Business Recommendations
def generate_executive_summary(df):
    """Generate comprehensive executive summary with actionable insights"""
    
    print("=" * 80)
    print("EXECUTIVE SUMMARY: VENDOR PERFORMANCE ANALYSIS")
    print("=" * 80)
    
    # Overall Business Metrics
    total_sales = df['TotalSalesDollars'].sum()
    total_purchases = df['TotalPurchaseDollars'].sum()
    total_profit = df['GrossProfit'].sum()
    overall_margin = (total_profit / total_sales) * 100
    total_vendors = df['VendorNumber'].nunique()
    total_brands = df['Brand'].nunique()
    
    print(f"\nBUSINESS OVERVIEW:")
    print(f"• Total Sales Revenue: ${total_sales:,.2f}")
    print(f"• Total Purchase Cost: ${total_purchases:,.2f}")
    print(f"• Gross Profit: ${total_profit:,.2f}")
    print(f"• Overall Profit Margin: {overall_margin:.2f}%")
    print(f"• Active Vendors: {total_vendors}")
    print(f"• Active Brands: {total_brands}")
    
    # Key Findings
    print(f"\nKEY FINDINGS:")
    
    # 1. Vendor Concentration
    top_10_vendors = df.groupby('VendorName')['TotalSalesDollars'].sum().sort_values(ascending=False).head(10)
    vendor_concentration = (top_10_vendors.sum() / total_sales) * 100
    print(f"1. VENDOR CONCENTRATION: Top 10 vendors contribute {vendor_concentration:.1f}% of total sales")
    
    # 2. Profit Margin Analysis
    avg_profit_margin = df['ProfitMargin'].mean()
    print(f"2. PROFIT MARGINS: Average profit margin is {avg_profit_margin:.2f}%")
    
    # 3. Inventory Efficiency
    avg_turnover = df['StockTurnover'].mean()
    print(f"3. INVENTORY TURNOVER: Average stock turnover is {avg_turnover:.2f}x")
    
    # 4. Unsold Inventory
    unsold_inventory = ((df['TotalPurchaseQuantity'] - df['TotalSalesQuantity']) * df['UnitCost']).sum()
    print(f"4. CAPITAL TIED UP: ${unsold_inventory:,.2f} in unsold inventory")
    
    # 5. Freight Costs
    total_freight = df['FreightCost'].sum()
    freight_ratio = (total_freight / total_purchases) * 100
    print(f"5. FREIGHT COSTS: ${total_freight:,.2f} total ({freight_ratio:.2f}% of purchases)")
    
    return {
        'total_sales': total_sales,
        'total_profit': total_profit,
        'vendor_concentration': vendor_concentration,
        'avg_profit_margin': avg_profit_margin,
        'unsold_inventory': unsold_inventory,
        'freight_ratio': freight_ratio
    }

exec_summary = generate_executive_summary(df_analysis)

In [ ]:
# Business Recommendations
def business_recommendations(df, summary):
    """Generate actionable business recommendations"""
    
    print("\n" + "=" * 80)
    print("STRATEGIC RECOMMENDATIONS")
    print("=" * 80)
    
    print(f"\n1. VENDOR RELATIONSHIP OPTIMIZATION")
    print(f"   • Strengthen relationships with top 10 vendors ({summary['vendor_concentration']:.1f}% of sales)")
    print(f"   • Negotiate better terms with high-volume vendors")
    print(f"   • Develop performance improvement plans for underperforming vendors")
    
    print(f"\n2. INVENTORY MANAGEMENT")
    print(f"   • Address ${summary['unsold_inventory']:,.2f} in unsold inventory")
    print(f"   • Implement just-in-time inventory for slow-moving products")
    print(f"   • Consider clearance sales for items with stock turnover < 1.0")
    
    print(f"\n3. PRICING STRATEGY")
    print(f"   • Review pricing for brands with low sales but high margins")
    print(f"   • Implement dynamic pricing based on inventory turnover")
    print(f"   • Leverage bulk purchasing discounts (up to 72% cost reduction observed)")
    
    print(f"\n4. OPERATIONAL EFFICIENCY")
    print(f"   • Optimize freight costs (currently {summary['freight_ratio']:.2f}% of purchases)")
    print(f"   • Consolidate shipments to reduce transportation costs")
    print(f"   • Evaluate alternative logistics providers")
    
    print(f"\n5. FINANCIAL PERFORMANCE")
    print(f"   • Target profit margin improvement from {summary['avg_profit_margin']:.2f}% to 35%+")
    print(f"   • Focus on high-margin product categories")
    print(f"   • Implement vendor scorecard for performance tracking")
    
    # Risk Assessment
    print(f"\n6. RISK MITIGATION")
    print(f"   • Diversify vendor base to reduce dependency on top suppliers")
    print(f"   • Establish backup suppliers for critical products")
    print(f"   • Implement vendor performance monitoring system")

business_recommendations(df_analysis, exec_summary)

In [ ]:
# Save Analysis Results
def save_analysis_results(df, summary):
    """Save analysis results for reporting and dashboard"""
    
    # Create comprehensive summary table
    vendor_summary = df.groupby('VendorName').agg({
        'TotalSalesDollars': 'sum',
        'TotalPurchaseDollars': 'sum',
        'GrossProfit': 'sum',
        'ProfitMargin': 'mean',
        'StockTurnover': 'mean',
        'TotalPurchaseQuantity': 'sum',
        'TotalSalesQuantity': 'sum',
        'FreightCost': 'sum'
    }).round(2)
    
    vendor_summary['SalesRank'] = vendor_summary['TotalSalesDollars'].rank(ascending=False)
    vendor_summary['ProfitRank'] = vendor_summary['GrossProfit'].rank(ascending=False)
    vendor_summary['TurnoverRank'] = vendor_summary['StockTurnover'].rank(ascending=False)
    
    # Save to CSV for Power BI
    vendor_summary.to_csv('vendor_performance_summary.csv')
    
    # Brand summary
    brand_summary = df.groupby('Description').agg({
        'TotalSalesDollars': 'sum',
        'TotalPurchaseDollars': 'sum',
        'GrossProfit': 'sum',
        'ProfitMargin': 'mean',
        'TotalSalesQuantity': 'sum'
    }).round(2)
    
    brand_summary.to_csv('brand_performance_summary.csv')
    
    print("Analysis results saved:")
    print("• vendor_performance_summary.csv")
    print("• brand_performance_summary.csv")
    
    return vendor_summary, brand_summary

vendor_summary, brand_summary = save_analysis_results(df_analysis, exec_summary)

print(f"\nFinal Dataset Summary:")
print(f"• {len(df_analysis)} vendor-brand combinations analyzed")
print(f"• {df_analysis['VendorName'].nunique()} unique vendors")
print(f"• {df_analysis['Description'].nunique()} unique brands")
print(f"• Analysis period: Full year 2024")
print(f"• Total business value: ${exec_summary['total_sales']:,.2f}")